# Character-Level GPT for Poetry

End-to-end notebook: corpus → tokenization → training → generation → analysis.

Each module is self-contained and documented. Read the `.py` files first;
this notebook just wires them together and runs the experiments.

In [ ]:
# If running on Colab, upload the .py files or clone your repo first.
# Then make sure dependencies are installed:
!pip install torch --quiet

## 1. Corpus

In [ ]:
## 1. Corpus

import os
import pickle
import random
from collections import defaultdict

import os

CORPUS_FILE = '/content/poetry_corpus.txt'  # uploaded file path in Colab

if os.path.exists(CORPUS_FILE):
    print("Loading corpus from local session...")
    with open(CORPUS_FILE, 'r', encoding='utf-8') as f:
        text = f.read()
    print(f"Corpus: {len(text):,} characters ({len(text)/1024/1024:.1f} MB)")
else:
    raise FileNotFoundError(
        "Corpus file not found. Upload 'poetry_corpus.txt' using the file sidebar or files.upload()."
    )


Loading corpus from local session...
Corpus: 7,798,473 characters (7.4 MB)


## 2. Tokenization

In [ ]:
from language_modeling import CharTokenizer, bits_per_character

tok = CharTokenizer(text)
print(f'Vocabulary size: {tok.vocab_size}')
print(f'Vocabulary: {tok.vocab}')

ids = tok.encode(text)
split = int(0.9 * len(ids))
train_ids, val_ids = ids[:split], ids[split:]
print(f'\nTrain tokens: {len(train_ids):,} | Val tokens: {len(val_ids):,}')

Vocabulary size: 127
Vocabulary: ['\t', '\n', ' ', '!', '"', '$', '&', "'", '(', ')', '*', '+', ',', '-', '.', '/', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', ':', ';', '<', '=', '>', '?', '@', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', '[', '\\', ']', '^', '_', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '{', '|', '}', '~', '\x9d', '£', '¤', '¶', 'Æ', 'È', 'Ô', 'à', 'á', 'â', 'ä', 'æ', 'ç', 'è', 'é', 'ê', 'ë', 'ì', 'í', 'î', 'ï', 'ñ', 'ò', 'ô', 'ö', 'ù', 'û', 'ü', 'ÿ', 'œ', '’', '“', '”']

Train tokens: 7,018,625 | Val tokens: 779,848


## 3. Model

In [ ]:
from model import CharGPT
import torch

CONTEXT_LENGTH = 256

model = CharGPT(
    vocab_size=tok.vocab_size,
    embed_dim=256,
    num_heads=8,
    num_layers=6,
    context_length=CONTEXT_LENGTH,
    dropout=0.1,
)
print(f'Parameters: {model.count_parameters():,}')

Parameters: 4,830,976


## 4. Training

In [ ]:
from training import train

model = train(
    model, train_ids, val_ids, tok,
    context_length=CONTEXT_LENGTH,
    batch_size=64,
    max_steps=20_000,
    max_lr=3e-4,
    eval_interval=500,
    sample_interval=2000,
)

Training on cuda. Steps: 20000, batch: 64
Train tokens: 7,018,625 | Val tokens: 779,848

step   100 | loss 3.1018 | bpc 4.4749 | lr 6.00e-05
step   200 | loss 2.5995 | bpc 3.7503 | lr 1.20e-04
step   300 | loss 2.4717 | bpc 3.5660 | lr 1.80e-04
step   400 | loss 2.4525 | bpc 3.5381 | lr 2.40e-04
step   500 | loss 2.4030 | bpc 3.4669 | lr 3.00e-04

  [val] step 500 | loss 2.5398 | bpc 3.6642

step   600 | loss 2.3863 | bpc 3.4427 | lr 3.00e-04
step   700 | loss 2.3267 | bpc 3.3566 | lr 3.00e-04
step   800 | loss 2.2129 | bpc 3.1926 | lr 3.00e-04
step   900 | loss 2.1340 | bpc 3.0787 | lr 3.00e-04
step  1000 | loss 2.0870 | bpc 3.0108 | lr 3.00e-04

  [val] step 1000 | loss 2.1933 | bpc 3.1642

step  1100 | loss 2.0304 | bpc 2.9293 | lr 2.99e-04
step  1200 | loss 1.9737 | bpc 2.8474 | lr 2.99e-04
step  1300 | loss 1.9839 | bpc 2.8622 | lr 2.99e-04
step  1400 | loss 1.8960 | bpc 2.7354 | lr 2.99e-04
step  1500 | loss 1.8618 | bpc 2.6860 | lr 2.98e-04

  [val] step 1500 | loss 1.9728 | bpc

## 5. Generation

In [ ]:
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Seed with a short prompt from the corpus
prompt_text = 'Shall I compare'
prompt_ids  = torch.tensor(tok.encode(prompt_text), dtype=torch.long).unsqueeze(0).to(device)

generated_ids = model.generate(
    prompt_ids,
    max_new_tokens=500,
    temperature=0.8,
    top_k=40,
)

generated_text = tok.decode(generated_ids[0].tolist())
print(generated_text)

Shall I compare my wonderful.
So grant me; then, thou time is but here,
Have proved thee, my own life from me,
When thou shalt have more dead at last!
Pan thy best will be but have me seen.
But when thy pilgrim flames the night
Reach me then light their flights fall,
The spirit I laugh in his right-hand,
And the spirit's country should hear him
Who needs not in the trumpet life.
The brisk enough the mother far
Should all do mine and thee
That came mine own thee a slave;
And shall be sweet or child.
Thou kissed


## 6. Structural Analysis

In [ ]:
from analysis import compare_corpora, probe_newline

compare_corpora(text, generated_text)

=== Structural Comparison ===

Line length   | train: mean=39.0, std=10.0, median=39.0
              | gen:   mean=33.4, std=7.4, median=37.0

Stanza length | train: mean=2464.8, std=4265.8, median=1052.0
              | gen:   mean=15.0, std=0.0, median=15.0

Rhyme density | train: 0.051
              | gen:   0.071


## 7. Probing

In [ ]:
# Use a held-out sequence for probing
probe_ids = torch.tensor(val_ids[:512], dtype=torch.long)
probe_newline(model, probe_ids, tok, device=device)

=== Probing: newline prediction by layer ===

Base rate (always predict majority): 0.977



AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
